# Aula 18 — Observabilidade: enxergar pai e filhos (com e sem LangSmith)

**Objetivo:** responder 'quem chamou quem, quanto tempo levou e onde quebrou'
para grafos com sub-grafos — localmente (sem chave) e na nuvem (LangSmith).

**Pré-requisito:** Aula 17 (pai que chama filhos).

**Como usar:** rode em ordem. Autocontido, 100% offline (a seção LangSmith é
só instrução: sem `LANGCHAIN_API_KEY` aqui). Comece pela CÉLULA 0.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph
print('dependências prontas.')

## 1. Analogia: painel de voos vs caixa-preta

Sem observabilidade, o grafo é caixa-preta: entra pedido, sai erro, ninguém
sabe onde. Com ela, é painel de voos: cada chamada com nome, duração e pai.

| Painel | No LangGraph | O que é de verdade |
|---|---|---|
| cada voo com origem | **`BaseCallbackHandler`** | recebe `on_chain_start/end/error` de tudo |
| 'voo filho do voo X' | **`parent_run_id`** | liga cada execução ao pai (a árvore) |
| duração do voo | **tempo entre start/end** | gargalo por nó, de graça |
| torre na nuvem | **LangSmith** | o MESMO protocolo, enviado para `smith.langchain.com` |

Ponto-chave: LangSmith não é outro sistema — é este callback com destino na
nuvem. Tudo que você constrói aqui funciona lá (e vice-versa).

In [ ]:
# CÉLULA 1 — o rastreador local (rode e leia a árvore)
import time
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage
from langgraph.graph import START, END, MessagesState, StateGraph

class RastreadorLocal(BaseCallbackHandler):
    def __init__(self):
        self.linhas = []
        self._inicio = {}; self._nomes = {}; self._prof = {}

    def on_chain_start(self, serialized, inputs, *, run_id, parent_run_id=None,
                       name=None, **kwargs):
        self._nomes[run_id] = name
        self._inicio[run_id] = time.perf_counter()
        self._prof[run_id] = self._prof.get(parent_run_id, -1) + 1

    def _fecha(self, run_id, status: str):
        dt = (time.perf_counter() - self._inicio.pop(run_id)) * 1000
        rec = '  ' * self._prof.get(run_id, 0) + str(self._nomes.get(run_id))
        self.linhas.append((rec, f'{dt:.1f}ms', status))

    def on_chain_end(self, outputs, *, run_id, **kwargs):
        self._fecha(run_id, 'ok')

    def on_chain_error(self, error, *, run_id, **kwargs):
        self._fecha(run_id, f'ERRO {type(error).__name__}')

def sub_grafinho(nome_no: str):
    b = StateGraph(MessagesState)
    b.add_node(nome_no, lambda s: {'messages': []})
    b.add_edge(START, nome_no); b.add_edge(nome_no, END)
    return b.compile()

pai = StateGraph(MessagesState)
pai.add_node('filho_a', sub_grafinho('trabalha_a'))
pai.add_node('filho_b', sub_grafinho('trabalha_b'))
pai.add_edge(START, 'filho_a'); pai.add_edge(START, 'filho_b')
pai.add_edge('filho_a', END); pai.add_edge('filho_b', END)
grafo_pai = pai.compile()

rast = RastreadorLocal()
list(grafo_pai.stream({'messages': [HumanMessage(content='oi')]}, config={'callbacks': [rast]}))
for nome, tempo, status in rast.linhas:
    print(f'{nome} [{tempo}] {status}')

## 2. Conceito: o protocolo de callbacks

- Toda execução LangChain/LangGraph emite eventos: `on_chain_start` (com `name`,
  `run_id`, `parent_run_id`), `on_chain_end` (com saídas) e `on_chain_error`.
- `parent_run_id` é o grampo da árvore: cada execução sabe quem a chamou.
  Profundidade = quantos pais acima (raiz = 0).
- `config={'callbacks': [rast]}` pluga o handler **numa execução** (não global).
  Sem callbacks, nada é registrado — observabilidade é opt-in.
- O `metadata` ainda carrega `langgraph_node`, `langgraph_step`, `langgraph_path`
  (Aula 11 usou para filtrar tokens; aqui usamos `name` + tempos).

In [ ]:
# CÉLULA 2 — o caminho do erro (rode e siga a subida)
def sub_quebrado(nome_no: str):
    def pane(state: MessagesState) -> dict:
        raise RuntimeError('pane no filho')
    b = StateGraph(MessagesState)
    b.add_node(nome_no, pane)
    b.add_edge(START, nome_no); b.add_edge(nome_no, END)
    return b.compile()

pai2 = StateGraph(MessagesState)
pai2.add_node('filho_ok', sub_grafinho('trabalha_ok'))
pai2.add_node('filho_ruim', sub_quebrado('trabalha_ruim'))
pai2.add_edge(START, 'filho_ok'); pai2.add_edge(START, 'filho_ruim')
pai2.add_edge('filho_ok', END); pai2.add_edge('filho_ruim', END)
grafo_erro = pai2.compile()

rast2 = RastreadorLocal()
try:
    list(grafo_erro.stream({'messages': [HumanMessage(content='oi')]}, config={'callbacks': [rast2]}))
except RuntimeError as erro:
    print(f'(propagou: {erro})')
for nome, tempo, status in rast2.linhas:
    print(f'{nome} [{tempo}] {status}')
print()
print('Leitura: o ERRO sobe (filho -> pai); o ramo ok continua ok.')

## 3. Dissecando a árvore (como ler cada linha)

- Indentação = profundidade (`parent_run_id` encadeado). Raiz (`LangGraph`)
  fecha por último — a lista é ordenada por **término**, não por início.
- `filho_ok`/`filho_ruim` são os **nós do pai**; `LangGraph` indentado abaixo
  deles é o **sub-grafo** executando; `trabalha_*` é o nó de dentro.
- Tempo por nível responde 'onde está lento?' sem profiler: o gargalo é o
  maior `ms` na árvore (aqui tudo é ~0ms; com LLM real, o `assistant` domina).

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. LangSmith: as mesmas 3 linhas, na nuvem

Sem `LANGCHAIN_API_KEY` aqui, então esta célula só **instrui** (não executa
nada remoto). Com a chave, o mesmo grafo das células acima traça sozinho.

In [ ]:
# CÉLULA 3 — LangSmith opcional (só instrução, sem rede)
import os

if os.getenv('LANGCHAIN_API_KEY'):
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ.setdefault('LANGCHAIN_PROJECT', 'reforco-langgraph')
    print('tracing ativo: https://smith.langchain.com (projeto reforco-langgraph)')
    print('rode qualquer célula de grafo de novo e veja a árvore na nuvem.')
else:
    print('sem LANGCHAIN_API_KEY: nada a fazer — o RastreadorLocal já mostrou tudo.')
    print('quando tiver a chave, exporte e rode o mesmo código:')
    print('  LANGCHAIN_TRACING_V2=true')
    print('  LANGCHAIN_API_KEY=<sua>')
    print('  LANGCHAIN_PROJECT=reforco-langgraph  (opcional)')
    print()
    print('Na UI você verá: a mesma árvore (pai -> filhos), tempos por nível,')
    print('inputs/outputs clicáveis e o caminho do erro em vermelho.')

### 4b. Auditoria complementar: `get_state_history` (Aula 15)

Callbacks mostram **como executou** (tempos, erros); checkpoints mostram **o
que resultou** (estados). Para debugar 'por que saiu X?', combine os dois:
o trace diz *onde*, o histórico diz *com que dados*.

In [ ]:
# CÉLULA 4 — trace + histórico juntos (rode e compare)
from langgraph.checkpoint.memory import MemorySaver

paudit = StateGraph(MessagesState)
paudit.add_node('filho_a', sub_grafinho('trabalha_a'))
paudit.add_edge(START, 'filho_a'); paudit.add_edge('filho_a', END)
gaudit = paudit.compile(checkpointer=MemorySaver())
cfga = {'configurable': {'thread_id': 'auditoria'}}
rasta = RastreadorLocal()
list(gaudit.stream({'messages': [HumanMessage(content='oi')]}, {**cfga, 'callbacks': [rasta]}))
print('trace diz COMO:')
for nome, tempo, status in rasta.linhas:
    print(f'  {nome} [{tempo}] {status}')
print('histórico diz COM QUE DADOS:')
for h in gaudit.get_state_history(cfga):
    print('  next =', h.next, '| msgs =', len(h.values['messages']))

## 5. Erros clássicos

1. **Esperar trace sem callbacks.** Sem `config={'callbacks': [...]}`, nada é
   registrado — observabilidade é opt-in por execução.
2. **Ler a lista em ordem de início.** Ela vem ordenada por **término** (filhos
   fecham antes dos pais). Para ordem de início, carimbe tempo no `start`.
3. **Achar que erro no filho morre nele.** Ele **propaga** até o pai (CÉLULA 2)
   e aborta o `stream` — capture fora se quiser continuar outros ramos.
4. **Confundir trace com estado.** Trace = como executou; `get_state_history` =
   com que dados. Um não substitui o outro (CÉLULA 4).
5. **Vazar a chave no `.env` versionado.** `LANGCHAIN_API_KEY` segue a mesma
   regra da `GROQ_API_KEY`: `.env` local, nunca no git (Aula 01).
6. **Rastrear tudo em produção.** Callback por execução tem custo (I/O, disco).
   Amostre (ex: 10%) ou ligue sob demanda em debug.

## 6. Exercício guiado — rastreie o salão da Aula 17

Situação: o salão (`cozinha` + `bar` + `fechar` da Aula 17) com o rastreador.
Valide que `fechar` aparece, que `cozinha`/`bar` são filhos dele... na verdade
não — valide a topologia real: quem é filho de quem na árvore.

In [ ]:
# CÉLULA 5 — exercício guiado (rastreie o salão)
from typing import TypedDict

class Salao(TypedDict):
    pedido: str
    prato: str
    bebida: str

bs = StateGraph(Salao)
bs.add_node('cozinha', lambda s: {'prato': f"prato({s['pedido']})"})
bs.add_node('bar', lambda s: {'bebida': f"bebida({s['pedido']})"})
bs.add_edge(START, 'cozinha'); bs.add_edge(START, 'bar')
bs.add_edge('cozinha', END); bs.add_edge('bar', END)
salaoc = bs.compile()

rastex = RastreadorLocal()
outx = salaoc.invoke({'pedido': 'pastel', 'prato': '', 'bebida': ''},
                     config={'callbacks': [rastex]})
for nome, tempo, status in rastex.linhas:
    print(f'{nome} [{tempo}] {status}')
nomes = [n.strip() for n, _, _ in rastex.linhas]
assert 'cozinha' in nomes and 'bar' in nomes
assert outx['prato'] == 'prato(pastel)'
print('OK — salão rastreado: cozinha e bar no mesmo nível (irmãos, não pai-filho).')

## 7. Desafio — caçador de gargalo

Adicione `time.sleep(0.2)` num dos filhos do salão e:

1. Rode com o rastreador e identifique o gargalo só pelos tempos.
2. Estenda o handler com `on_llm_start/on_llm_end` para separar tempo de LLM
   de tempo de tool no mesmo nó.
3. **Pegadinha:** `time.sleep` bloqueia o event loop — em `astream` use
   `asyncio.sleep`. O rastreador mede espera como 'tempo do nó': saiba ler.

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] O que liga uma execução ao pai — e como se calcula a profundidade?
- [ ] Por que a lista vem ordenada por término (e o que isso muda na leitura)?
- [ ] Erro no filho: onde ele aparece e até onde sobe?
- [ ] Trace vs checkpoints: o que cada um responde?
- [ ] LangSmith exige mudar o código? (Se não, o que exige?)
- [ ] No seu avaliador, o que você rastrearia por etapa (tempo de LLM vs regra)?

**Próxima aula (19 — MapReduce, última do curso):** N tarefas dinâmicas com
`Send` (map) + síntese (reduce) — paralelismo que nasce em runtime.